# LLM で日本語の文法誤りを訂正して評価する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_gec_education/gec_ja_llm.ipynb)

NLP クックブック 27-07 ／ 文法誤り訂正と言語教育

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

日本語学習者の書いた文の誤りを訂正したい。LLM（GPU を使う）に訂正させ、参照訂正と比べて評価する。

In [1]:
!pip install -q datasets

In [2]:
import difflib, re
import sentencepiece as spm
import torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる (LLM の章の「オープンな LLM とチャットする」を参照)。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

@torch.no_grad()
def chat(prompt, max_new_tokens=80):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip().split("\n")[0]

def clean(text):
    """LLM が付け足しがちな「訂正後の文:」のような見出しや、末尾の注釈 (「（誤りはありません）」) を取り除く"""
    text = re.sub(r"^(訂正後の文|訂正文|書き直した文|文)\s*[:：]\s*", "", text.strip())
    return re.sub(r"[（(][^（）()]*[）)]$", "", text).strip().strip("「」")

MINIMAL = ("次の文は日本語学習者が書いたものです。文法の誤りだけを、できるだけ少ない変更で訂正してください。"
           "誤りがなければそのまま出力してください。訂正後の文だけを 1 行で出力し、説明は書かないでください。\n\n文: {}")
FLUENT = "次の文は日本語学習者が書いたものです。自然で流暢な日本語に書き直してください。書き直した文だけを 1 行で出力してください。\n\n文: {}"

for s in ["図書館で本が借りました。", "私は毎日学校を行きます。", "昨日は友達と映画を見ます。", "この料理はとてもおいしいでした。", "私は東京に住んでいます。"]:
    print(f"{s}\n   最小限の訂正: {clean(chat(MINIMAL.format(s)))}\n   流暢な書き直し: {clean(chat(FLUENT.format(s)))}")

図書館で本が借りました。
   最小限の訂正: 図書館で本を借りました。
   流暢な書き直し: 図書館で本を借りました。
私は毎日学校を行きます。
   最小限の訂正: 私は毎日学校に行きます。
   流暢な書き直し: 私は毎日学校に通っています。
昨日は友達と映画を見ます。
   最小限の訂正: 昨日、友達と映画を見に行きました。」が正しい文です。訂正点は「見ます」を「見に行きました」に変更したことです。これは、映画を見るという動作が完了したことを表すために「行く」の過去形「行きました」を使用したものです。
   流暢な書き直し: 昨日は友達と映画を見に行きました。
この料理はとてもおいしいでした。
   最小限の訂正: この料理はとても美味しいでした。
   流暢な書き直し: この料理はとても美味しかったです。
私は東京に住んでいます。
   最小限の訂正: 私は東京に住んでいます。
   流暢な書き直し: 私は東京に住んでいます。


In [3]:
# 訂正を編集 (どこをどう変えたか) の集合にして、参照訂正と比べる
def edits(source, target):
    """文字単位の差分から (開始位置, 終了位置, 置き換える文字列) の集合を作る"""
    ops = difflib.SequenceMatcher(None, source, target, autojunk=False).get_opcodes()
    return {(i1, i2, target[j1:j2]) for op, i1, i2, j1, j2 in ops if op != "equal"}

def evaluate(system, n=100):
    ds = load_dataset("ronantakizawa/jfleg-japanese", split="validation").select(range(n))
    tp = fp = fn = exact = unchanged = 0
    examples = []
    for ex in ds:
        hyp = system(ex["sentence"])
        hyp_edits = edits(ex["sentence"], hyp)
        # 参照訂正が複数あるので、最も F0.5 が高くなるものと比べる
        best = max((edits(ex["sentence"], ref) for ref in ex["corrections"]),
                   key=lambda ref: (1.25 * len(hyp_edits & ref) / (len(hyp_edits) + 0.25 * len(ref) + 1e-9), -len(ref)))
        tp += len(hyp_edits & best); fp += len(hyp_edits - best); fn += len(best - hyp_edits)
        exact += hyp in ex["corrections"]; unchanged += hyp == ex["sentence"]
        examples.append((ex["sentence"], hyp, ex["corrections"][0]))
    p, r = tp / (tp + fp + 1e-9), tp / (tp + fn + 1e-9)
    return {"適合率": p, "再現率": r, "F0.5": 1.25 * p * r / (0.25 * p + r + 1e-9), "参照と完全一致": exact / n, "変更なし": unchanged / n}, examples

import pandas as pd

# 100 文 × 2 通りの指示で生成する。T4 で 5 分ほど
results, outputs = {}, {}
for label, system in [("何もしない", lambda s: s), ("LLM (最小限の訂正)", lambda s: clean(chat(MINIMAL.format(s)))), ("LLM (流暢な書き直し)", lambda s: clean(chat(FLUENT.format(s))))]:
    results[label], outputs[label] = evaluate(system)
pd.DataFrame(results).T.round(3)

,適合率,再現率,F0.5,参照と完全一致,変更なし
何もしない,0.000,0.000,0.000,0.01,1.00
LLM (最小限の訂正),0.455,0.392,0.441,0.24,0.17
LLM (流暢な書き直し),0.418,0.519,0.435,0.19,0.01


In [4]:
for src, hyp, ref in outputs["LLM (最小限の訂正)"][:8]:
    print(f"{'○' if hyp == ref else '　'} {src}\n     → {hyp}\n     参照: {ref}")

○ 図書館で本が借りました。
     → 図書館で本を借りました。
     参照: 図書館で本を借りました。
　 公園に友達と遊びました。
     → 公園に友達と遊びました。
     参照: 公園で友達と遊びました。
　 私の家を駅の近くあります。
     → 私の家は駅のの近くにあります。
     参照: 私の家は駅の近くにあります。
　 彼は公園を走っている。
     → 彼は公園を走っています。
     参照: 彼は公園で走っている。
○ 駅で電車を乗りました。
     → 駅で電車に乗りました。
     参照: 駅で電車に乗りました。
　 仕事が終わった後、映画館に行きます。
     → 仕事が終わった後、映画館に行きます。
     参照: 仕事が終わったら、映画館に行きます。
　 子供たちを公園へ遊びに行きました。
     → 子供たちを公園へ遊びに行きました。
     参照: 子供たちを公園に遊びに行きました。
　 レストランが夕食を食べました。
     → レストランは夕食を食べませんでした。
     参照: レストランで夕食を食べました。


- 「誤りだけを最小限に直す」のか「自然な文に書き直す」のかで、出力も評価も大きく変わる。指示で明確にする。学習者へのフィードバックには前者、文章の清書には後者が向く。
- 出力の整形が評価を左右する。小さな LLM は「訂正後の文:」のような見出しや「（誤りはありません）」のような注釈を付けたり、助詞を重ねて（「駅のの近く」）新しい誤りを作ったりする。見出しや注釈は機械的に取り除き、新しい誤りはそのまま評価に反映させる。訂正文に続けて説明を書き出す出力（「…見に行きました。」が正しい文です。…」）は `clean` では取り除けないので、必要なら規則を足す。
- 意味を変えてしまう訂正も出る（「レストランが夕食を食べました」→「レストランは夕食を食べませんでした」）。
- LLM は直しすぎる傾向がある（正しい部分まで言い換える）。編集単位の適合率が下がるので、F0.5 のように適合率を重視する指標では不利になる。この実験では、流暢な書き直しは再現率が高い (0.519) が適合率が低く (0.418)、F0.5 は 0.435 で、最小限の訂正の 0.441 をわずかに下回った（T4 と同じ float16 で実行した値。100 文なので、この程度の差は誤差の範囲）。逆に、最小限の訂正は直すべき文を変えずに残すことがある（「公園に友達と遊びました。」）。
- 参照訂正は 1 つとは限らない。複数の参照と比べないと、正しい訂正を誤りと数えてしまう。それでも参照にない正しい訂正は残る。
- ここでは文字単位の差分で編集を作った。きちんと評価するなら、形態素単位で編集を取り出す（この章の ERRANT のレシピの日本語版にあたる処理）。
- このデータは JFLEG にならって作られた公開データで、文が短く誤りも単純。実際の学習者の作文（Lang-8 コーパス、TEC-JL など）はもっと難しい。